# 04 · Validate — geometry preservation, MD-THERMOSTABILITY ranking, docking & track comparison

**Standard slot:** *validate (in silico).* **For Project 19 the benchmark has two headlines:** the
**catalytic-geometry preservation rate** and the **MD-based thermostability ranking** (RMSF +
melting-proxy) — the project's emphasis. Plus **PET-mimic pocket accessibility** (docking) and the
**engineered-natural vs fully de novo** scaffold comparison `[extension]` (D3 pt2).

Needs `results/campaign.csv` (+ `results/ranked.csv` from notebook 03).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Catalytic-geometry preservation — the geometry headline
Distribution of catalytic-geometry RMSD vs the 0.5 Å pass bar. The fraction left of the line is the
**preservation rate**. (Numbers here are SYNTHETIC mock values; on Colab they come from real AF2
predictions.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")
cut = 0.5

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(camp["catalytic_geom_rmsd"], bins=20)
ax.axvline(cut, color="k", ls="--", lw=1, label=f"pass < {cut} A")
ax.set_xlabel("catalytic-geometry RMSD vs theozyme (A)  [SYNTHETIC]")
ax.set_ylabel("designs"); ax.set_title("Catalytic-geometry preservation (triad + oxyanion hole)")
ax.legend(); plt.tight_layout()
plt.savefig("results/catalytic_geometry_hist.png", dpi=150); plt.show()

rate = 100 * (camp["catalytic_geom_rmsd"] <= cut).mean()
print(f"overall catalytic-geometry preservation rate = {rate:.1f}%  [SYNTHETIC demo]")

## 2 · MD-based thermostability ranking — THE PROJECT EMPHASIS
Rank the **geometry-passing** survivors by the MD thermostability proxy: **low catalytic/backbone
RMSF + a high melting-proxy** ⇒ more likely to survive industrial temperatures (~65-70 °C). This is a
**proxy, not a Tm** — DSF (notebook 05) gives the real number. Mind the **thermostability ↔ activity
trade-off**: do not pick only the most rigid designs.

In [ ]:
geom_ok = camp[camp["catalytic_geom_rmsd"] <= 0.5].copy()
ranked_thermo = geom_ok.sort_values("thermostability_rank_score", ascending=False)
ranked_thermo.to_csv("results/thermostability_ranked.csv", index=False)

fig, ax = plt.subplots(figsize=(5.6, 3.4))
sc = ax.scatter(ranked_thermo["catalytic_rmsf"], ranked_thermo["melting_proxy"],
                c=ranked_thermo["catalytic_geom_rmsd"], cmap="viridis")
ax.set_xlabel("active-site (catalytic) RMSF (A) — lower = more rigid  [SYNTHETIC]")
ax.set_ylabel("melting-proxy (0-1) — higher = more thermostable  [SYNTHETIC]")
ax.set_title("Thermostability ranking of geometry-passing designs")
fig.colorbar(sc, label="catalytic-geom RMSD (A)")
plt.tight_layout(); plt.savefig("results/thermostability_ranking.png", dpi=150); plt.show()

print(f"{len(ranked_thermo)} geometry-passing designs ranked by thermostability [SYNTHETIC].")
print("Top-left + high (rigid active site, high melting-proxy) = most thermostable; but beware the")
print("thermostability <-> activity trade-off — a too-rigid active site can be catalytically dead.")
print("\nTop 5 by thermostability proxy:")
print(ranked_thermo[["design_id", "track", "catalytic_geom_rmsd", "catalytic_rmsf",
                     "melting_proxy", "thermostability_rank_score"]].head().to_string(index=False))

## 3 · Engineered-natural vs fully de novo scaffolds `[extension]`
Compare the two tracks: which holds the triad geometry *and* stays thermostable? In the real campaign
the engineered-natural track grafts the triad onto a stable cutinase/IsPETase scaffold; the de novo
track builds the fold from scratch. Here both tracks may be present if you ran them in nb02; otherwise
this cell shows the *shape* of the comparison you will populate on Colab.

In [ ]:
by_track = (camp.assign(pass_geom=camp["catalytic_geom_rmsd"] <= 0.5)
                .groupby("track")
                .agg(n=("design_id", "size"),
                     geom_pass_rate=("pass_geom", "mean"),
                     mean_melting_proxy=("melting_proxy", "mean"),
                     mean_catalytic_rmsf=("catalytic_rmsf", "mean"),
                     mean_plddt_cat=("plddt_catalytic", "mean"))
                .reset_index())
by_track["geom_pass_rate"] = (100 * by_track["geom_pass_rate"]).round(1)
print("Track comparison (engineered-natural vs fully de novo) [SYNTHETIC]:")
print(by_track.to_string(index=False))
print("\n[SYNTHETIC] On Colab: run BOTH tracks on the SAME theozyme and compare geometry-preservation")
print("AND thermostability — the engineered-natural track often wins on stability, the de novo track")
print("on novelty. Report the trade-off honestly.")

## 4 · PET-mimic substrate docking + active-site stability (top candidates)
Docking (AutoDock Vina) checks the **PET-mimic ester fits and is oriented** toward Ser-OG — pocket
accessibility, not affinity, not activity. Cross it against the thermostability proxy for the ranked
survivors as orthogonal evidence.

In [ ]:
# merge ranked.csv (fp.Design fields) with campaign.csv docking/thermo fields by design_id
try:
    ranked = pd.read_csv("results/ranked.csv")
    ranked = ranked.merge(
        camp[["design_id", "vina_score", "pose_in_pocket", "oriented_to_ser",
              "melting_proxy", "catalytic_rmsf"]],
        on="design_id", how="left")
except FileNotFoundError:
    ranked = camp.copy()

top = ranked.head(min(20, len(ranked)))
fig, ax = plt.subplots(figsize=(5.4, 3.4))
sc = ax.scatter(top["vina_score"], top["melting_proxy"],
                c=top["catalytic_rmsf"], cmap="plasma")
ax.set_xlabel("Vina PET-mimic fit score (more negative = better fit)  [SYNTHETIC]")
ax.set_ylabel("melting-proxy (higher = more thermostable)  [SYNTHETIC]")
ax.set_title("Top candidates: pocket fit vs thermostability")
fig.colorbar(sc, label="catalytic RMSF (A)")
plt.tight_layout(); plt.savefig("results/docking_thermostability.png", dpi=150); plt.show()
print("Lower-left-to-upper (good fit + high melting-proxy + low RMSF) are the best candidates [SYNTHETIC].")

## 5 · Honest hit-rate accounting
Report N(pass all layers) / N(generated), and remind the reader of the field reality: even a good
preservation rate + a good thermostability proxy is **not** an activity rate or a measured Tm.
Geometry ≠ catalysis; an MD proxy ≠ thermostability; activity (pNP-ester / PET-film) + DSF are required.

In [ ]:
n_total = len(camp)
try:
    ranked = pd.read_csv("results/ranked.csv")
    n_hits = int((ranked["layers_passed"] >= 3).sum())
except Exception:
    n_hits = int((camp["catalytic_geom_rmsd"] <= 0.5).sum())
n_thermo = int(((camp["catalytic_geom_rmsd"] <= 0.5) & (camp["melting_proxy"] >= 0.6)).sum())
print("Hit-rate accounting [SYNTHETIC demo]:")
print(f"  generated                          : {n_total}")
print(f"  pass all filter layers             : {n_hits}  ({100*n_hits/max(n_total,1):.1f}%)")
print(f"  geometry-pass AND thermostable proxy: {n_thermo}  ({100*n_thermo/max(n_total,1):.1f}%)")
print("\nREALITY CHECK: de novo enzyme activity rates are <5% without directed evolution; in-silico")
print("catalytic geometry does NOT guarantee activity, and an MD proxy does NOT guarantee real")
print("thermostability. Only an activity assay + DSF decide. Mind the thermostability<->activity trade-off.")

## D3 (part 2) checklist
- [ ] Catalytic-geometry preservation histogram (`results/catalytic_geometry_hist.png`) + rate.
- [ ] **MD-thermostability ranking** figure (`results/thermostability_ranking.png`) + `thermostability_ranked.csv`.
- [ ] Engineered-natural vs fully de novo track comparison `[extension]`.
- [ ] PET-mimic docking + thermostability figure on the ranked top set.
- [ ] Honest hit-rate accounting with the "geometry ≠ activity" + "MD proxy ≠ Tm" caveats stated.

**Next:** `05_validation_plan.ipynb` — the activity + DSF assay plan + controls + surface-redesign stretch.